# Kiểm tra dữ liệu FinSurvival – nghi vấn trùng dòng khi nối X và y

Đặt notebook này trong `D:\BlockChain\DMLR_DeFi_Survival_Benchmark` (cùng chỗ với notebook EDA) rồi **Run All**.

**Nghi vấn:** notebook EDA nối X và y bằng cột `id` (`merge`). Nếu một `id` xuất hiện nhiều lần (nhiều lần vay trong cùng một giao dịch), phép nối sẽ **nhân bản dòng** → tổng số dòng sau khi gắn nhãn lớn hơn bộ gốc (264.605 > 264.536).

**Notebook kiểm tra 6 điều:**
1. Số dòng X và y có bằng nhau, có khớp bài báo (264.536) không
2. `id` có bị trùng trong X / trong y không
3. X và y có cùng thứ tự dòng không (để ghép theo vị trí)
4. Nối bằng `merge(on="id")` sinh thêm bao nhiêu dòng
5. Gắn nhãn theo cách đúng (ghép theo vị trí) cho ra bao nhiêu dòng, % lớp 1 – so với bài báo (255.209 dòng, 4,77%)
6. File `train_labeled` / `test_labeled` hiện tại có bị nhân bản không

Cuối notebook có ô **lưu lại dữ liệu đã sửa** – mặc định TẮT (`SAVE_FIXED = False`), chỉ bật sau khi đọc kết quả.

In [9]:
import os
import numpy as np
import pandas as pd

P = "processed"
T = 17.43 * 86400          # ngưỡng 17,43 ngày (giây)
PAPER_RECORDS, PAPER_LABELLED, PAPER_POS = 264_536, 255_209, 4.77

X = {s: pd.read_parquet(f"{P}/X_{s}.parquet") for s in ["train", "test"]}
y = {s: pd.read_parquet(f"{P}/y_{s}_liquidated.parquet") for s in ["train", "test"]}
for s in X:
    print(f"{s}: X {X[s].shape} | y {y[s].shape}")

train: X (161826, 129) | y (161826, 5)
test: X (98060, 129) | y (102710, 5)


## 1. Số dòng X và y

In [10]:
n = {s: (len(X[s]), len(y[s])) for s in X}
tot_x = sum(v[0] for v in n.values()); tot_y = sum(v[1] for v in n.values())
display(pd.DataFrame(n, index=["X", "y"]).T)
print(f"Tổng X = {tot_x:,} | tổng y = {tot_y:,} | bài báo = {PAPER_RECORDS:,}")
print("-> KHỚP bài báo" if tot_x == tot_y == PAPER_RECORDS else "-> KHÔNG khớp, cần xem lại file đã tải")

,X,y
train,161826,161826
test,98060,102710


Tổng X = 259,886 | tổng y = 264,536 | bài báo = 264,536
-> KHÔNG khớp, cần xem lại file đã tải


## 2. `id` có bị trùng không

In [11]:
rows = []
for s in X:
    dx, dy = X[s]["id"].duplicated(keep=False), y[s]["id"].duplicated(keep=False)
    rows.append({"tập": s,
                 "id duy nhất trong X": X[s]["id"].nunique(),
                 "dòng có id trùng (X)": int(dx.sum()),
                 "dòng có id trùng (y)": int(dy.sum()),
                 "id lặp nhiều nhất (lần)": int(X[s]["id"].value_counts().max())})
display(pd.DataFrame(rows).set_index("tập"))

# Các dòng trùng id có giống hệt nhau không, hay là các lần vay khác nhau trong cùng giao dịch?
s = "train"
dup_ids = X[s]["id"][X[s]["id"].duplicated()].unique()
if len(dup_ids):
    ex = X[s][X[s]["id"] == dup_ids[0]]
    cols = [c for c in ["id", "user", "reserve", "amountUSD", "timestamp"] if c in ex.columns]
    print(f"\nVí dụ một id bị trùng ({len(ex)} dòng):")
    display(ex[cols])
    full_dup = X[s].duplicated(keep=False).sum()
    print("Số dòng trùng HOÀN TOÀN (mọi cột giống nhau):", int(full_dup))
else:
    print("Không có id trùng trong train.")

,id duy nhất trong X,dòng có id trùng (X),dòng có id trùng (y),id lặp nhiều nhất (lần)
tập,,,,
train,160051,2892,2892,17
test,97288,1267,5917,13



Ví dụ một id bị trùng (2 dòng):


,id,user,reserve,amountUSD,timestamp
636,0x70da097e8a72703a073c8200e04fcb8609df88a1a98a...,0xa4292fec84e6bed56b21e49d22b5e3e927f7078a,USDC,66513.164342,1.607433e+09
637,0x70da097e8a72703a073c8200e04fcb8609df88a1a98a...,0xa4292fec84e6bed56b21e49d22b5e3e927f7078a,WETH,9174.056740,1.607433e+09


Số dòng trùng HOÀN TOÀN (mọi cột giống nhau): 0


## 3. X và y có cùng thứ tự dòng không

In [12]:
same_order = {}
for s in X:
    same_order[s] = len(X[s]) == len(y[s]) and bool((X[s]["id"].values == y[s]["id"].values).all())
    print(f"{s}: cùng thứ tự = {same_order[s]}")
print("\n-> Có thể ghép X và y THEO VỊ TRÍ (an toàn)" if all(same_order.values())
      else "\n-> KHÔNG cùng thứ tự: không ghép theo vị trí được, báo lại để xử lý cách khác")

train: cùng thứ tự = False
test: cùng thứ tự = False

-> KHÔNG cùng thứ tự: không ghép theo vị trí được, báo lại để xử lý cách khác


## 4. Nối bằng `merge(on='id')` sinh thêm bao nhiêu dòng

In [13]:
for s in X:
    m = X[s][["id"]].merge(y[s][["id"]], on="id")
    print(f"{s}: trước {len(X[s]):,} -> sau merge {len(m):,}  (thêm {len(m) - len(X[s]):,} dòng do nhân bản)")

train: trước 161,826 -> sau merge 169,104  (thêm 7,278 dòng do nhân bản)
test: trước 98,060 -> sau merge 100,442  (thêm 2,382 dòng do nhân bản)


## 5. Thống kê nhãn chỉ từ y (không cần ghép với X)

Số dòng và % lớp 1 ở đây chỉ dựa vào y nên luôn đúng, dù cách ghép với X có vấn đề.

In [14]:
def label_of(yy):
    lab = pd.Series(np.nan, index=yy.index)
    lab[(yy["status"] == 1) & (yy["timeDiff"] <= T)] = 1
    lab[yy["timeDiff"] > T] = 0
    return lab

fixed, rows = {}, []
for s in X:
    df = y[s][["id", "timeDiff", "status"]].reset_index(drop=True)
    df["label"] = label_of(df)
    keep = df["label"].notna()
    fixed[s] = df[keep].reset_index(drop=True)
    rows.append({"tập": s, "ban đầu": len(df), "bị bỏ": int((~keep).sum()), "giữ lại": int(keep.sum()),
                 "lớp 1": int((df["label"] == 1).sum()), "% lớp 1": round(100 * df.loc[keep, "label"].mean(), 2)})
tab = pd.DataFrame(rows).set_index("tập")
tab.loc["tổng"] = tab.sum()
tab.loc["tổng", "% lớp 1"] = round(100 * tab.loc["tổng", "lớp 1"] / tab.loc["tổng", "giữ lại"], 2)
display(tab)
print(f"Bài báo: {PAPER_LABELLED:,} dòng, {PAPER_POS}% lớp 1 (bài báo còn bỏ thêm vùng đệm 30 ngày)")

,ban đầu,bị bỏ,giữ lại,lớp 1,% lớp 1
tập,,,,,
train,161826.0,4185.0,157641.0,10889.0,6.91
test,102710.0,1912.0,100798.0,1273.0,1.26
tổng,264536.0,6097.0,258439.0,12162.0,4.71


Bài báo: 255,209 dòng, 4.77% lớp 1 (bài báo còn bỏ thêm vùng đệm 30 ngày)


## 6. File `train_labeled` / `test_labeled` hiện tại có bị nhân bản không

In [15]:
for s in ["train", "test"]:
    f = f"{P}/{s}_labeled.parquet"
    if not os.path.exists(f):
        print(f"{f}: chưa có"); continue
    cur = pd.read_parquet(f, columns=["id", "label"])
    print(f"{s}_labeled hiện tại: {len(cur):,} dòng, {100 * cur['label'].mean():.2f}% lớp 1 "
          f"| theo y: {len(fixed[s]):,} dòng, {100 * fixed[s]['label'].mean():.2f}% "
          f"-> {'OK' if len(cur) == len(fixed[s]) else 'LỆCH ' + format(len(cur) - len(fixed[s]), ',') + ' dòng'}")

train_labeled hiện tại: 164,819 dòng, 6.69% lớp 1 | theo y: 157,641 dòng, 6.91% -> LỆCH 7,178 dòng
test_labeled hiện tại: 99,786 dòng, 1.30% lớp 1 | theo y: 100,798 dòng, 1.26% -> LỆCH -1,012 dòng


## 7. Kiểm tra file X_test có thiếu không và cách tác giả ghép dữ liệu

X_test ít hơn y_test vài nghìn dòng: kiểm tra xem đã tải đủ các phần `X_test_*.rds` chưa, và xem `dataLoader.R` của tác giả ghép X với y thế nào.

In [16]:
import glob
base = os.path.join("Data", "Survival_Data", "Borrow")
for folder in ["X_train", "X_test"]:
    files = sorted(glob.glob(os.path.join(base, folder, "*.rds")))
    print(folder, "->", [f"{os.path.basename(f)} ({os.path.getsize(f) / 1e6:.1f} MB)" for f in files])
print()
print(open("dataLoader.R", encoding="utf-8", errors="ignore").read())

X_train -> ['X_train_1.rds (43.0 MB)', 'X_train_2.rds (43.2 MB)', 'X_train_3.rds (43.2 MB)', 'X_train_4.rds (43.2 MB)']
X_test -> ['X_test_1.rds (34.8 MB)', 'X_test_2.rds (34.9 MB)', 'X_test_3.rds (34.9 MB)']

# Author: Aaron Green
library(readr)
library(stringr)
library(dplyr)


## Helper functions
not_all_na <- function(x) any(!is.na(x))
`%notin%` <- Negate(`%in%`)


loadSurvivalDataset <- function(indexEvent, outcomeEvent, 
                                dataPath = "~/KDD_DeFi_Survival_Dataset_And_Benchmark/Data/Survival_Data/", 
                                X_path = "/X_train/",
                                y_path = "y_train.rds"){
  
  
  X_files <- list.files(paste0(dataPath, str_to_title(indexEvent), X_path), pattern = NULL, all.files = FALSE, full.names = TRUE)
  X = data.frame()
  
  for(file in X_files){
    X <- X %>%
      bind_rows(read_rds(file)) %>%
      select(where(not_all_na)) %>%
      select(-starts_with("exo")) %>%
      filter(!is.na(id))
  }
  
  y <- rea

## 8. Cách sửa đúng: bỏ các id bị trùng rồi ghép 1–1 theo id

X và y **không cùng thứ tự**, nên KHÔNG ghép theo vị trí được. y cũng không có cột nào (như `reserve`) để phân biệt các khoản vay chung một `id`. Cách an toàn: **bỏ các dòng có `id` trùng** ở cả X và y (các khoản vay này không xác định được nhãn), rồi ghép theo `id` với kiểm tra 1–1 (`validate="one_to_one"`), mỗi khoản vay chỉ còn đúng 1 dòng.

In [17]:
fixed2, rows = {}, []
for s in X:
    xs = X[s][~X[s]["id"].duplicated(keep=False)]
    ys = y[s][~y[s]["id"].duplicated(keep=False)]
    df = xs.merge(ys[["id", "timeDiff", "status"]], on="id", how="inner", validate="one_to_one")
    df["label"] = label_of(df)
    keep = df["label"].notna()
    fixed2[s] = df[keep].reset_index(drop=True)
    rows.append({"tập": s, "X ban đầu": len(X[s]), "bỏ do id trùng": len(X[s]) - len(xs),
                 "không có trong y": len(xs) - len(df), "bỏ do chưa đủ 17,43 ngày": int((~keep).sum()),
                 "giữ lại": int(keep.sum()), "% lớp 1": round(100 * df.loc[keep, "label"].mean(), 2)})
tab2 = pd.DataFrame(rows).set_index("tập")
display(tab2)
tot = tab2["giữ lại"].sum()
pos = sum((fixed2[s]["label"] == 1).sum() for s in fixed2)
print(f"Tổng giữ lại: {tot:,} dòng, {100 * pos / tot:.2f}% lớp 1 (bài báo: {PAPER_LABELLED:,} dòng, {PAPER_POS}%)")

,X ban đầu,bỏ do id trùng,không có trong y,"bỏ do chưa đủ 17,43 ngày",giữ lại,% lớp 1
tập,,,,,,
train,161826,2892,0,4108,154826,6.98
test,98060,1267,0,448,96345,1.31


Tổng giữ lại: 251,171 dòng, 4.81% lớp 1 (bài báo: 255,209 dòng, 4.77%)


## 9. Lưu dữ liệu đã sửa (chỉ chạy sau khi đọc kết quả mục 7–8)

Đổi `SAVE_FIXED = True` rồi chạy ô này. File cũ được đổi tên thành `*_OLD.parquet`.
Sau đó chạy lại: `python make_dataset_figures.py` → notebook 02.

In [19]:
SAVE_FIXED = True

if SAVE_FIXED:
    for s in fixed2:
        f = f"{P}/{s}_labeled.parquet"
        if os.path.exists(f):
            os.replace(f, f"{P}/{s}_labeled_OLD.parquet")
        out = fixed2[s].copy()
        out["label"] = out["label"].astype(int)
        out.to_parquet(f)
        print("Đã lưu", f, out.shape)
else:
    print("Chưa lưu. Đọc kết quả mục 7-8, rồi đặt SAVE_FIXED = True và chạy lại ô này.")

Đã lưu processed/train_labeled.parquet (154826, 132)
Đã lưu processed/test_labeled.parquet (96345, 132)
